# 第 1 章　Python 环境配置

> **T1 共三章**
> - **第 1 章（本章）**：Python 环境配置
> - 第 2 章：工具链（Git / Marp / Quarto / Stata×Python）
> - 第 3 章：金融数据 API 获取

::: {.callout-note}
## 本章要点

1. 用 Anaconda 创建课程专用虚拟环境 `dsfin`
2. 安装全课程所需的 Python 库
3. 把 `dsfin` 注册为 Jupyter kernel，让所有 notebook 都能选到它
4. 验证安装：一键检查所有必要库是否就绪

本章是**纯配置章**，内容少但必须做对——
环境配置出问题会让后面每一章的代码都跑不了。
遇到问题优先查本章末尾的「常见问题」一节。
:::


---

## 0　为什么用虚拟环境

你的电脑上可能同时有多个项目，每个项目依赖的库版本不同。
如果所有项目共用同一套 Python 环境，迟早会遇到
「升级 A 项目的库导致 B 项目崩溃」的问题。

**虚拟环境**让每个项目有自己独立的 Python 和库，互不干扰。
本课程使用 **Anaconda** 管理虚拟环境，原因：
- `conda` 能同时管理 Python 版本和系统级依赖
- 提供图形界面（Anaconda Navigator），对不熟悉命令行的同学友好
- 与 `pip` 完全兼容

如果你已经安装了 Anaconda，直接跳到第 2 节。


---

## 1　安装 Anaconda

从 [https://www.anaconda.com/download](https://www.anaconda.com/download) 下载，
选择对应你操作系统的版本（Python 3.11 以上）。

**安装时的两个关键选项：**

- **「Add Anaconda to PATH」**：
  Windows 用户**建议勾选**（方便在普通终端里用 `conda` 命令）；
  macOS/Linux 安装程序会自动配置，无需手动勾选。
- **「Register Anaconda as my default Python」**：建议勾选。

安装完成后，打开终端（macOS/Linux）或 **Anaconda Prompt**（Windows），
运行下面的命令验证：

```bash
conda --version    # 应输出类似 conda 24.x.x
python --version   # 应输出类似 Python 3.11.x
```


---

## 2　创建课程专用虚拟环境

在终端或 Anaconda Prompt 里，**按顺序**执行下面的命令。
整个过程大约需要 5–10 分钟（取决于网络速度）。

```bash
# Step 1：创建名为 dsfin 的虚拟环境，指定 Python 3.11
conda create -n dsfin python=3.11 -y

# Step 2：激活环境（每次打开新终端都需要执行这一步）
conda activate dsfin

# Step 3：安装核心分析库
pip install pandas numpy matplotlib seaborn scipy statsmodels

# Step 4：安装机器学习库
pip install scikit-learn xgboost lightgbm shap

# Step 5：安装金融数据 API 库
pip install akshare fredapi wbgapi yfinance tushare

# Step 6：安装回归与因果推断库
pip install statsmodels linearmodels pyfixest doubleml

# Step 7：安装 Jupyter 和 NLP 库
pip install jupyterlab ipykernel
pip install jieba gensim transformers sentence-transformers

# Step 8：安装可视化扩展
pip install plotly mplfinance quantstats

# Step 9：把 dsfin 注册为 Jupyter kernel
# 这样在 JupyterLab 里新建 notebook 时能选到「dsfin（课程环境）」
python -m ipykernel install --user --name dsfin --display-name 'dsfin（课程环境）'
```

::: {.callout-warning}
## 最常见的错误：忘记先激活环境

如果没有先运行 `conda activate dsfin` 就执行 `pip install`，
库会装到 `base` 环境，不是 `dsfin`。
后果：在 dsfin kernel 里 `import akshare` 会报 `ModuleNotFoundError`。

**验证方法**：激活后运行 `which python`（macOS/Linux）
或 `where python`（Windows），路径里应包含 `dsfin`。
:::

### 2.1　国内网络的安装加速

如果 `pip install` 速度很慢，可以临时使用清华镜像源：

```bash
# 临时使用清华镜像（-i 只对本次命令生效）
pip install akshare -i https://pypi.tuna.tsinghua.edu.cn/simple

# 永久设置（对所有后续 pip install 生效）
pip config set global.index-url https://pypi.tuna.tsinghua.edu.cn/simple
```


---

## 3　验证安装

安装完成后，在 JupyterLab 里选择 `dsfin（课程环境）` kernel，
运行下面的 cell。所有库都显示 `✓` 才算配置成功。


In [ ]:
# ── 3.1  环境验证 ────────────────────────────────────────────────────────
# 运行这个 cell，确认所有必要的库都已正确安装。
# 如有 ✗ 或 ⚠，按提示重新安装对应的库。

import sys
import importlib

required = {
    # 核心分析
    'pandas'        : '2.0',
    'numpy'         : '1.24',
    'matplotlib'    : '3.7',
    'seaborn'       : '0.12',
    'scipy'         : '1.10',
    'statsmodels'   : '0.14',
    # 机器学习
    'sklearn'       : '1.3',
    'xgboost'       : '1.7',
    'lightgbm'      : '4.0',
    # 金融数据
    'akshare'       : '1.0',
    'fredapi'       : '0.5',
    # 回归
    'linearmodels'  : '5.3',
    'pyfixest'      : '0.18',
    # 可视化
    'plotly'        : '5.0',
}

print(f'Python {sys.version.split()[0]}  |  环境：{sys.prefix}\n')
print(f'{'库':<16} {'版本':<12} {'最低要求':<10} 状态')
print('─' * 52)

all_ok = True
for pkg, min_ver in required.items():
    try:
        mod = importlib.import_module(pkg)
        ver = getattr(mod, '__version__', '?')
        ok  = ver >= min_ver if ver != '?' else True
        flag = '✓' if ok else '⚠ 版本过低'
        if not ok:
            all_ok = False
    except ImportError:
        ver, flag, all_ok = '未安装', '✗', False
    print(f'{pkg:<16} {ver:<12} {min_ver:<10} {flag}')

print()
if all_ok:
    print('✅  环境配置完成，所有库就绪。')
else:
    print('⚠️  部分库缺失或版本过低，请按第 2 节重新安装后再次运行。')


---

## 4　Jupyter 的常用技巧

### 4.1　快捷键

按 `Esc` 进入命令模式，按 `Enter` 进入编辑模式。

| 操作 | 命令模式快捷键 |
|------|--------------|
| 运行当前 cell，移到下一个 | `Shift + Enter` |
| 运行当前 cell，留在原地 | `Ctrl + Enter` |
| 在上方插入 cell | `A` |
| 在下方插入 cell | `B` |
| 删除当前 cell | `DD`（连按两次 D）|
| 转为 Markdown cell | `M` |
| 转为 Code cell | `Y` |
| 查找和替换 | `Ctrl + H` |

### 4.2　魔法命令

```python
%timeit df.groupby('stkcd').mean()    # 计时单行代码
%%time                                 # 计时整个 cell
%matplotlib inline                    # 图表直接显示在 notebook 里
%load_ext autoreload                  # 自动重新加载修改过的模块
%autoreload 2
```

### 4.3　全局显示设置

每章第一个 code cell 都包含下面这段全局设置，
确保中文字体正常、浮点数格式统一。
你在自己的 notebook 里可以直接复制使用。


In [ ]:
# ── 4.3  全局显示设置（每章第一个 code cell 的标准写法）──────────────────
import warnings
import matplotlib
import pandas as pd

warnings.filterwarnings('ignore')

# 中文字体：按系统可用字体依次尝试
matplotlib.rcParams['font.sans-serif'] = ['SimHei', 'Arial Unicode MS', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False

# DataFrame 显示设置
pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.4f}'.format)

print('全局显示设置已应用 ✓')


---

## 5　常见问题

**Q：运行验证 cell 时出现 `ModuleNotFoundError: No module named 'akshare'`**

A：说明库装到了错误的环境。检查步骤：
①在终端运行 `conda activate dsfin`；
②运行 `pip install akshare`；
③在 JupyterLab 里确认 kernel 选的是「dsfin（课程环境）」（右上角显示）。

---

**Q：中文在图表里显示为方块**

A：字体未找到。在 code cell 开头加：
```python
import matplotlib.pyplot as plt
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei']  # Windows
# plt.rcParams['font.sans-serif'] = ['PingFang SC']   # macOS
```
或者安装思源字体后重启 Jupyter。

---

**Q：`conda activate dsfin` 在 Windows 命令提示符里报错**

A：使用 **Anaconda Prompt**（不是普通的 cmd 或 PowerShell）。
如果一定要用 PowerShell，先运行 `conda init powershell`，然后重启 PowerShell。

---

**Q：安装时提示「SSL certificate verify failed」**

A：临时绕过 SSL 验证（仅限内网或可信网络环境）：
```bash
pip install akshare --trusted-host pypi.tuna.tsinghua.edu.cn \
    -i https://pypi.tuna.tsinghua.edu.cn/simple
```

::: {.callout-tip}
## 提示词示范（环境报错）

````
我在运行 Jupyter Notebook 时遇到以下报错：
[粘贴完整的错误信息，包括 Traceback]

我的环境：
- 操作系统：[Windows 11 / macOS 14 / Ubuntu 22.04]
- Python 版本：[python --version 的输出]
- 虚拟环境：[conda info --envs 的输出]
- 相关库版本：[pip show 库名 的输出]

请帮我：
1. 解释这个错误的原因
2. 给出最小可复现的修复步骤
3. 说明如何验证修复是否成功
````
:::
